## Arcface loss

In [1]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [2]:
# Parameters
query_ratio = 0.2
seed = 4


In [3]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict
import sys
sys.path.append(r'C:\BP\pythonProject1')
from misclassification_utils import show_misclassified

In [4]:

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (319, 768)


In [5]:
encoder = joblib.load(encoder_path)

# Convert names back later:
names = encoder.inverse_transform(labels)


In [6]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [7]:
if __name__ == "__main__":
    import os, random, numpy as np, torch, torch.nn.functional as F
    from sklearn.model_selection import train_test_split  # (can remove)
    from sklearn.neighbors import KNeighborsClassifier
    import torch.nn as nn
    import torch.optim as optim
    import pandas as pd
    from proportional_split_xy import proportional_split_xy

    # ---------- reproducibility ----------
    seed = seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # ---------- use existing embeddings and labels ----------

    embeddings_tensor = torch.from_numpy(embeddings).float()
    N, D = embeddings_tensor.shape
    print("Using embeddings:", embeddings_tensor.shape)

    labels_tensor = torch.from_numpy(labels).long()
    assert labels_tensor.shape[0] == N, f"Labels length {labels_tensor.shape[0]} != embeddings {N}"
    print("Using labels:", labels_tensor.shape, "num_classes:", len(torch.unique(labels_tensor)))

    # ---------- proportional split ----------
    X_train, X_test, y_train, y_test = proportional_split_xy(embeddings_tensor.numpy(), labels_tensor.numpy(), query_ratio=0.2)
    print("Train/Test sizes:", len(X_train), len(X_test))

    # Convert to torch tensors
    X_train = torch.from_numpy(np.array(X_train)).float().to(device)
    y_train = torch.from_numpy(np.array(y_train)).long().to(device)
    X_test = torch.from_numpy(np.array(X_test)).float().to(device)
    y_test = torch.from_numpy(np.array(y_test)).long().to(device)

    # ---------- ArcFace head implementation ----------
    class ArcFaceHead(nn.Module):
        def __init__(self, in_features, out_features, s=30.0, m=0.5):
            super().__init__()
            self.s = s
            self.m = m
            self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
            nn.init.xavier_uniform_(self.weight)

        def forward(self, x, labels=None):
            x_norm = F.normalize(x, p=2, dim=1)
            W = F.normalize(self.weight, p=2, dim=1)
            cosine = torch.matmul(x_norm, W.t()).clamp(-1.0, 1.0)
            if labels is None:
                return cosine * self.s
            theta = torch.acos(cosine)
            target_logits = torch.cos(theta + self.m)
            one_hot = torch.zeros_like(cosine)
            one_hot.scatter_(1, labels.view(-1, 1), 1.0)
            logits = cosine * (1 - one_hot) + target_logits * one_hot
            logits = logits * self.s
            loss = F.cross_entropy(logits, labels)
            return loss, logits

    class HeadModel(nn.Module):
        def __init__(self, input_dim, feat_dim=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, feat_dim),
                nn.BatchNorm1d(feat_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(feat_dim, feat_dim // 2),
                nn.BatchNorm1d(feat_dim // 2),
                nn.ReLU(inplace=True),
            )
        def forward(self, x):
            return self.net(x)

    num_classes = int(len(torch.unique(labels_tensor)))
    feat_dim = 256
    head = HeadModel(D, feat_dim=feat_dim).to(device)
    arc = ArcFaceHead(in_features=feat_dim//2, out_features=num_classes, s=30.0, m=0.5).to(device)

    optimizer = optim.AdamW(list(head.parameters()) + list(arc.parameters()), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)

    # ---------- training ----------
    num_epochs = 100
    batch_size = 16
    best_test_acc = 0.0
    best_state = None

    for epoch in range(num_epochs + 1):
        head.train()
        arc.train()
        if epoch != 0:
            
            perm = torch.randperm(X_train.size(0), device=device)
            total_loss = 0.0
            total_samples = 0

            for i in range(0, len(perm), batch_size):
                batch_ids = perm[i:i+batch_size]
                batch_emb = X_train[batch_ids]
                batch_labels = y_train[batch_ids]

                optimizer.zero_grad()
                features = head(batch_emb)
                loss, _ = arc(features, batch_labels)
                loss.backward()
                optimizer.step()

                total_loss += float(loss.item()) * batch_emb.size(0)
                total_samples += batch_emb.size(0)

            avg_loss = total_loss / total_samples

            # ---------- validation ----------
            head.eval()
            arc.eval()
        with torch.no_grad():
            test_emb = head(X_test)
            logits = torch.matmul(F.normalize(test_emb, p=2, dim=1),
                                  F.normalize(arc.weight, p=2, dim=1).t()) * arc.s
            preds = logits.argmax(dim=1)
            test_acc = (preds == y_test).float().mean().item()

        if epoch == 0:
            print(f"Epoch {epoch:02d} | Test acc: {test_acc:.4f}")
        else:
            print(f"Epoch {epoch:02d} | Train loss: {avg_loss:.4f} | Test acc: {test_acc:.4f}")
        if epoch != 0:
            scheduler.step(avg_loss)

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state = {
                "head": head.state_dict(),
                "arc": arc.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "test_acc": test_acc,
            }
            torch.save(best_state, "best_arcface_state.pt")

    print("\n✅ Best test accuracy:", best_test_acc)

    if best_state is not None:
        head.load_state_dict(best_state["head"])
        arc.load_state_dict(best_state["arc"])

    head.eval()
    with torch.no_grad():
        train_calibrated = head(X_train)
        train_calibrated = F.normalize(train_calibrated, p=2, dim=1).cpu()
        test_calibrated = head(X_test)
        test_calibrated = F.normalize(test_calibrated, p=2, dim=1).cpu()

    torch.save(train_calibrated, "emb_arcface_train_calibrated.pt")
    torch.save(test_calibrated, "emb_arcface_test_calibrated.pt")
    print("Saved calibrated embeddings: emb_arcface_train_calibrated.pt and emb_arcface_test_calibrated.pt")

    # ---------- KNN classification with cosine similarity ----------
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn.fit(train_calibrated.numpy(), y_train.cpu().numpy())
    preds = knn.predict(test_calibrated.numpy())
    accuracy = (preds == y_test.cpu().numpy()).mean()
    print(f"KNN accuracy on calibrated embeddings: {accuracy:.4f}")

    torch.save({"head": head.state_dict(), "arc": arc.state_dict()}, "arcface_model_final.pt")
    print("Saved model state: arcface_model_final.pt")


Device: cuda
Using embeddings: torch.Size([319, 768])
Using labels: torch.Size([319]) num_classes: 14
Train/Test sizes: 250 69


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 00 | Test acc: 0.0580
Epoch 01 | Train loss: 17.6529 | Test acc: 0.4638


Epoch 02 | Train loss: 13.5101 | Test acc: 0.5217
Epoch 03 | Train loss: 11.3080 | Test acc: 0.5942
Epoch 04 | Train loss: 9.4895 | Test acc: 0.6087
Epoch 05 | Train loss: 8.1981 | Test acc: 0.5942
Epoch 06 | Train loss: 6.7349 | Test acc: 0.5507


Epoch 07 | Train loss: 5.7628 | Test acc: 0.5942
Epoch 08 | Train loss: 5.3081 | Test acc: 0.5942
Epoch 09 | Train loss: 4.3715 | Test acc: 0.5362
Epoch 10 | Train loss: 3.7048 | Test acc: 0.5362
Epoch 11 | Train loss: 3.5277 | Test acc: 0.6087


...

Epoch 77 | Train loss: 0.3309 | Test acc: 0.6087
Epoch 78 | Train loss: 0.0795 | Test acc: 0.6087
Epoch 79 | Train loss: 0.1633 | Test acc: 0.6087
Epoch 80 | Train loss: 0.1429 | Test acc: 0.6087
Epoch 81 | Train loss: 0.1097 | Test acc: 0.6087


Epoch 82 | Train loss: 0.1297 | Test acc: 0.6087
Epoch 83 | Train loss: 0.1193 | Test acc: 0.6087
Epoch 84 | Train loss: 0.1408 | Test acc: 0.5942
Epoch 85 | Train loss: 0.1734 | Test acc: 0.5942
Epoch 86 | Train loss: 0.1142 | Test acc: 0.5942


Epoch 87 | Train loss: 0.1085 | Test acc: 0.5942
Epoch 88 | Train loss: 0.0945 | Test acc: 0.5942
Epoch 89 | Train loss: 0.0775 | Test acc: 0.5942
Epoch 90 | Train loss: 0.1002 | Test acc: 0.5942
Epoch 91 | Train loss: 0.1137 | Test acc: 0.5942


Epoch 92 | Train loss: 0.2912 | Test acc: 0.5942
Epoch 93 | Train loss: 0.1323 | Test acc: 0.5942
Epoch 94 | Train loss: 0.1591 | Test acc: 0.5942
Epoch 95 | Train loss: 0.0405 | Test acc: 0.5942
Epoch 96 | Train loss: 0.0650 | Test acc: 0.5942


Epoch 97 | Train loss: 0.1089 | Test acc: 0.5942
Epoch 98 | Train loss: 0.0648 | Test acc: 0.5942
Epoch 99 | Train loss: 0.0485 | Test acc: 0.5942
Epoch 100 | Train loss: 0.0246 | Test acc: 0.5942

✅ Best test accuracy: 0.6376811861991882
Saved calibrated embeddings: emb_arcface_train_calibrated.pt and emb_arcface_test_calibrated.pt
KNN accuracy on calibrated embeddings: 0.5942
Saved model state: arcface_model_final.pt


In [8]:
result = {
    "megadescriptor_version": megadescriptor_version,
    "dataset_version": detection,
    "seed": seed,
    "query_ratio": query_ratio,
    "accuracy": accuracy,
    "loss_function": "ArcFace"
}

result

{'megadescriptor_version': 'T-224',
 'dataset_version': '',
 'seed': 4,
 'query_ratio': 0.2,
 'accuracy': 0.5942028985507246,
 'loss_function': 'ArcFace'}